<a href="https://colab.research.google.com/github/haricharansai/Deep-Learning-With-Accelerated-Computing-2.2/blob/main/Skill%20-%204.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Repository, Git Workflow and PyTorch–CUDA Development Environment Setup

This notebook prepares a beginner-friendly Python project repository and checks the tools needed for PyTorch development. It covers:

- Python, operating-system, Git, and NVIDIA driver checks
- A practical project directory structure
- Safe Git identity, commit, branch, remote, and push workflows
- Linux/Colab and Windows virtual-environment commands
- PyTorch installation guidance using the official selector
- CUDA/PyTorch verification, a GPU tensor test, and memory reporting

> **Important:** Run cells from top to bottom. Some cells create files or change the notebook process's working directory. If you restart the kernel, rerun the setup cells.

## 0. Hosted Linux versus local Windows

A traceback containing `/tmp/ipykernel...` and Python 3.13 usually means this notebook is running in a hosted Linux environment such as Colab or a remote Jupyter server. That runtime is a different computer from a local Windows laptop.

- A hosted runtime can use only the GPU assigned to that runtime. It cannot directly access a local RTX 4050.
- To use the RTX 4050, launch Jupyter on the Windows computer and run this notebook there, with an NVIDIA driver installed.
- Linux commands below are for the hosted runtime. Windows commands are shown separately and should be run in PowerShell, Command Prompt, or Windows Terminal.
- Do not activate a virtual environment from a notebook cell and expect the already-running kernel to change. Create/activate the environment in a terminal, then start Jupyter from that environment.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

print('Python version :', sys.version.split()[0])
print('Python executable:', sys.executable)
print('Operating system:', platform.platform())
print('Architecture    :', platform.machine())
print('Current folder  :', Path.cwd())
print('Git on PATH     :', shutil.which('git') or 'not found')
print('nvidia-smi on PATH:', shutil.which('nvidia-smi') or 'not found')

Python version : 3.13.18
Python executable: /usr/bin/python3
Operating system: Linux-6.6.159+-x86_64-with-glibc2.39
Architecture    : x86_64
Current folder  : /content
Git on PATH     : /usr/bin/git
nvidia-smi on PATH: not found


In [ ]:
def run_command(args, *, cwd=None, check=False):
    """Run a command and print captured output instead of hiding useful errors."""
    print(f'\n$ {" ".join(map(str, args))}')
    try:
        result = subprocess.run(args, cwd=cwd, capture_output=True, text=True, check=False)
    except FileNotFoundError:
        print('Command not found. Install it or add it to PATH.')
        return None
    if result.stdout.strip(): print(result.stdout.strip())
    if result.stderr.strip(): print('STDERR:', result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {args}')
    return result

run_command(['git', '--version'])
gpu_check = run_command(['nvidia-smi'])
if gpu_check is None or gpu_check.returncode != 0:
    print('No usable NVIDIA runtime was detected here. This is normal for CPU-only hosted runtimes.')


$ git --version
git version 2.43.0

$ nvidia-smi
Command not found. Install it or add it to PATH.
No usable NVIDIA runtime was detected here. This is normal for CPU-only hosted runtimes.


## 1. Choose and create the project directory

Change `PROJECT_NAME` if desired. The code uses a path relative to the folder from which Jupyter was started, so it works in both hosted Linux and local Windows environments.

In [ ]:
PROJECT_NAME = 'PyTorch_CUDA_Project'
BASE_DIR = Path.cwd()
PROJECT_DIR = BASE_DIR / PROJECT_NAME
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)
print('Project directory:', PROJECT_DIR.resolve())

Project directory: /content/PyTorch_CUDA_Project


In [ ]:
folders = ['data/raw', 'data/processed', 'notebooks', 'src', 'models', 'logs', 'tests', 'configs']
for folder in folders:
    Path(folder).mkdir(parents=True, exist_ok=True)

# Keep otherwise-empty directories visible to Git.
for folder in folders:
    keep = Path(folder) / '.gitkeep'
    keep.touch(exist_ok=True)

print('Created project structure:')
for path in sorted(Path('.').rglob('*')):
    print('  ', path)

Created project structure:
   configs
   configs/.gitkeep
   data
   data/processed
   data/processed/.gitkeep
   data/raw
   data/raw/.gitkeep
   logs
   logs/.gitkeep
   models
   models/.gitkeep
   notebooks
   notebooks/.gitkeep
   src
   src/.gitkeep
   tests
   tests/.gitkeep


In [ ]:
gitignore = '''# Python
__pycache__/
*.py[cod]
.venv/
venv/
.env

# Jupyter
.ipynb_checkpoints/

# Data, models, logs, and local secrets
data/*
!data/raw/.gitkeep
!data/processed/.gitkeep
models/*
!models/.gitkeep
logs/*
!logs/.gitkeep
*.pth
*.pt
*.ckpt
secrets/
'''
Path('.gitignore').write_text(gitignore, encoding='utf-8')
requirements = '''numpy
matplotlib
jupyterlab
ipykernel
'''
Path('requirements.txt').write_text(requirements, encoding='utf-8')
Path('README.md').write_text('# PyTorch CUDA Project\n\nEnvironment setup and experiments.\n', encoding='utf-8')
print('Wrote .gitignore, requirements.txt, and README.md')

Wrote .gitignore, requirements.txt, and README.md


## 2. Git initialization and identity

The next cells are safe to rerun. Git identity is stored in Git configuration, not in the repository files. The prompts use placeholders only; never paste a GitHub password or personal access token into this notebook.

If commit fails with `Author identity unknown`, enter the requested name and email. If it fails with `nothing to commit`, the files may already be committed; inspect `git status`.

In [ ]:
run_command(['git', 'init'])
run_command(['git', 'status', '--short'])

git_name = input('Git author name (leave blank to skip): ').strip()
git_email = input('Git author email (leave blank to skip): ').strip()
if git_name:
    run_command(['git', 'config', 'user.name', git_name], check=True)
if git_email:
    run_command(['git', 'config', 'user.email', git_email], check=True)
print('Configured identity:')
run_command(['git', 'config', '--get', 'user.name'])
run_command(['git', 'config', '--get', 'user.email'])


$ git init
Initialized empty Git repository in /content/PyTorch_CUDA_Project/.git/
STDERR: hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>

$ git status --short
?? .gitignore
?? README.md
?? configs/
?? logs/
?? models/
?? notebooks/
?? requirements.txt
?? src/
?? tests/


In [ ]:
run_command(['git', 'add', '--all'], check=True)
commit = run_command(['git', 'commit', '-m', 'Initial project setup'])
if commit is not None and commit.returncode != 0:
    print('Commit did not complete. Read the captured error above.')
    print('Common fixes: configure user.name/user.email, or check whether there is nothing new to commit.')
run_command(['git', 'status', '--short'])
run_command(['git', 'log', '--oneline', '-1'])

## 3. Configure the GitHub remote and `main` branch

This is the requested remote URL:

`https://github.com/haricharansai/Deep-Learning-With-Accelerated-Computing-2.2`

The code checks whether `origin` already exists. It adds it when missing and updates it when present. Pushing may ask for GitHub authentication; use Git Credential Manager, SSH, or a short-lived token through Git's normal authentication prompt. Do not put credentials in notebook source.

In [ ]:
REMOTE_URL = 'https://github.com/haricharansai/Deep-Learning-With-Accelerated-Computing-2.2'
origin = run_command(['git', 'remote', 'get-url', 'origin'])
if origin is None or origin.returncode != 0:
    run_command(['git', 'remote', 'add', 'origin', REMOTE_URL], check=True)
else:
    run_command(['git', 'remote', 'set-url', 'origin', REMOTE_URL], check=True)
run_command(['git', 'remote', '-v'])
run_command(['git', 'branch', '-M', 'main'], check=True)
run_command(['git', 'branch', '--show-current'])

### Optional push

Run this only after confirming the remote repository is yours and authentication is ready. If the remote already contains commits, pull/reconcile them first rather than overwriting history.

In [ ]:
PUSH_NOW = False
if PUSH_NOW:
    run_command(['git', 'push', '-u', 'origin', 'main'])
else:
    print("Push skipped. When ready, run: git push -u origin main")

## 4. Virtual environments

Run these commands in a terminal, not in a running notebook kernel. After activation, install `requirements.txt` and launch Jupyter from the same environment.

### Hosted Linux / Ubuntu / Colab-style terminal
```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
jupyter lab
```

### Local Windows PowerShell
```powershell
py -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
jupyter lab
```

If PowerShell blocks activation, use `Set-ExecutionPolicy -Scope CurrentUser RemoteSigned` only if that policy change is acceptable on your machine, or use Command Prompt: `.\.venv\Scripts\activate.bat`.

## 5. Install PyTorch using the official selector

PyTorch's supported install commands change over time. Use the official selector instead of copying an old CUDA wheel command:

[PyTorch Start Locally](https://pytorch.org/get-started/locally/)

Choose your operating system, Pip, Python, and the CUDA option offered by the selector for your NVIDIA driver. Run the generated command in the activated environment's terminal. The notebook deliberately does not hardcode a CUDA build.

For a hosted runtime, install the command provided by that environment or its documented GPU image. A hosted runtime's CUDA availability is independent of the local RTX 4050.

In [ ]:
try:
    import torch
    print('PyTorch version:', torch.__version__)
    print('Built with CUDA :', torch.version.cuda)
    print('CUDA available  :', torch.cuda.is_available())
    print('GPU count       :', torch.cuda.device_count())
    if torch.cuda.is_available():
        print('GPU name        :', torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch is not installed in this notebook kernel.')
    print('Install it from the official selector, then restart the kernel.')

In [ ]:
try:
    import torch
    if not torch.cuda.is_available():
        print('GPU tensor test skipped: CUDA is unavailable in this runtime.')
    else:
        device = torch.device('cuda')
        x = torch.randn((2048, 2048), device=device)
        y = torch.randn((2048, 2048), device=device)
        z = x @ y
        torch.cuda.synchronize()
        print('Tensor device:', z.device)
        print('Tensor shape :', tuple(z.shape))
        print('Finite result:', bool(torch.isfinite(z).all()))
except ImportError:
    print('Install PyTorch before running the GPU tensor test.')

In [ ]:
try:
    import torch
    if torch.cuda.is_available():
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        print(f'Free GPU memory : {free_bytes / 1024**3:.2f} GiB')
        print(f'Total GPU memory: {total_bytes / 1024**3:.2f} GiB')
        print(f'Allocated       : {torch.cuda.memory_allocated() / 1024**2:.1f} MiB')
        print(f'Reserved        : {torch.cuda.memory_reserved() / 1024**2:.1f} MiB')
    else:
        print('Memory report skipped: CUDA is unavailable.')
except ImportError:
    print('Install PyTorch before requesting a memory report.')

## 6. Final environment summary

This final cell gathers the main facts needed when reporting setup issues. Save its output with any bug report.

In [ ]:
print('=== Environment summary ===')
print('Project directory:', Path.cwd().resolve())
print('Python:', sys.version.split()[0])
print('OS:', platform.platform())
git_result = run_command(['git', '--version'])
run_command(['git', 'branch', '--show-current'])
run_command(['git', 'remote', 'get-url', 'origin'])
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('Torch CUDA build:', torch.version.cuda)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
except ImportError:
    print('PyTorch: not installed in this kernel')
print('===========================')

## Troubleshooting quick reference

- **`Author identity unknown`:** configure `git config user.name` and `git config user.email`, then rerun `git add --all` and commit.
- **`nothing to commit`:** run `git status`; the initial commit may already exist.
- **`origin already exists`:** the notebook uses `git remote set-url origin ...` to update it safely.
- **`nvidia-smi` not found:** the runtime has no visible NVIDIA driver/tool, or this is a CPU-only hosted session.
- **`torch.cuda.is_available()` is `False`:** verify that the notebook kernel is the intended environment, install the PyTorch build from the official selector, restart the kernel, and verify the NVIDIA driver.
- **Local RTX 4050 is not visible in Colab/hosted Jupyter:** run Jupyter locally on Windows or attach a GPU runtime supplied by the hosting service.